In [ ]:
import torch
import numpy as np
from torch.nn import *
from liger_kernel.transformers import LigerRMSNorm
from liger_kernel.transformers.swiglu import LigerSiLUMulFunction
from liger_kernel.transformers.fused_linear_cross_entropy import LigerFusedLinearCrossEntropyFunction
from flash_attn import flash_attn_qkvpacked_func, flash_attn_kvpacked_func
from functools import reduce
from nn import Plot
torch._dynamo.config.cache_size_limit = 64
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class T(Sequential):
    def forward(s, _):
        return super().forward(_.mT).mT
class F(Module):
    def __init__(s, _):
        super().__init__()
        s._ = _
    def forward(s, _):
        return s._(_)
class Attn(Linear):
    def __init__(s, i, h=0, G=1, g=1, r=(7,7), drop=0., alibi=1.):
        h = h or i
        super().__init__(i, G*(g+2)*h)
        s._ = Linear(G*g*h, i, 0)
        s.__dict__ |= locals()
    def forward(s, _):
        _ = super().forward(_)
        return s._((flash_attn_qkvpacked_func if s.g<2 else flash_attn_kvpacked_func)(
            *[_.unflatten(-1,(3,-1,s.h))]if s.g<2 else[_[...,:-2*s.G*s.h].unflatten(-1,(-1,s.h)),_[...,:-2*s.G*s.h].unflatten(-1,(2,-1,s.h))],
            dropout_p=s.drop,
            window_size=s.r,
            alibi_slopes=torch.arange(s.G*s.g*s.alibi, device='cuda'),
        ).flatten(-2))
class SwiGLU(Linear):
    def __init__(s, i, h=0):
        super().__init__(i, 2*(h or i))
        s._ = Linear(h or i, i, 0)
    def forward(s, _):
        return s._(LigerSiLUMulFunction.apply(*super().forward(_).chunk(2,-1)))
def At(i, *_):
    return Res(RMSNorm(i), Attn(i, *_))
def Ff(i, *_):
    return Res(RMSNorm(i), SwiGLU(i, *_))
def U(i, *_):
    return Sequential(
        Linear(i, i*2, 0),
        Res(
            T(MaxPool1d(2)),
            Linear(i*2, i, 0),
            *_,
            Linear(i, i*4),
            Unflatten(2, (2,-1)),
            Flatten(1, 2),
        ),
        T(Conv1d(i*2, i, 3, padding=1, bias=0)),
    )
def entropy(p, n):
    return (np.log((n-1)/(n-2))+(-2*p).log1p()-(-p).log1p())*(n-1)
A = np.lib.stride_tricks.sliding_window_view(np.memmap(f'../../py256.bin', 'H'), 8192)

In [ ]:
i,h,G,g,a = 256,32,8,1,1024
m = Sequential(
    Embedding(256, i),
    At(i, h, G, g), Ff(i, a),
    U(i,
        At(i, h, G, g), Ff(i, a),
        U(i,
            At(i, h, G, g), Ff(i, a),
            U(i,
                At(i, h, G, g), Ff(i, a),
                U(i,
                    At(i, h, G, g), Ff(i, a),
                    U(i,
                        At(i, h, G, g), Ff(i, a),
                        U(i,
                            At(i, h, G, g), Ff(i, a),
                            U(i,
                                At(i, h, G, g), Ff(i, a),
                                U(i,
                                    At(i, h, G, g), Ff(i, a),
                                ),
                                At(i, h, G, g), Ff(i, a),
                            ),
                            At(i, h, G, g), Ff(i, a),
                        ),
                        At(i, h, G, g), Ff(i, a),
                    ),
                    At(i, h, G, g), Ff(i, a),
                ),
                At(i, h, G, g), Ff(i, a),
            ),
            At(i, h, G, g), Ff(i, a),
        ),
        At(i, h, G, g), Ff(i, a),
    ),
    At(i, h, G, g), Ff(i, a),
).to('cuda',torch.bfloat16)
m = torch.compile(m, dynamic=False)
try:
    _,o,O = torch.load('11g.pk', weights_only=False)
    m.load_state_dict(_)
except FileNotFoundError:
    o = torch.optim.Adam(m.parameters(), eps=1e-15)
    O = torch.optim.lr_scheduler.OneCycleLR(o, .0002, 100000, pct_start=.01, div_factor=10, final_div_factor=10)
plot = Plot(2, 10, 10)
try:
    while O.last_epoch < O.total_steps:
        t = torch.tensor(A[np.random.randint(0,len(A),8,'I')], dtype=torch.long, device='cuda')
        x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(256,[t.numel()//2],device='cuda'))
        y = m(x)
        l = LigerFusedLinearCrossEntropyFunction.apply(y.flatten(0,1), getattr(m,'0').weight, t.ravel())[0]
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        plot(l, lambda:entropy((-l).float().exp(), 256))
except KeyboardInterrupt:
    pass
torch.save((m.state_dict(),o,O), '11g.pk')

In [ ]:
l = a = 0
with torch.no_grad():
    for t in range(100):
        t = torch.tensor(A[np.random.randint(0,len(A),8,'I')], dtype=torch.long, device='cuda')
        x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(256,[t.numel()//2],device='cuda'))
        y = m(x)
        t = LigerFusedLinearCrossEntropyFunction.apply(y.flatten(0,1), getattr(m,'0').weight, t.ravel())[0]
        l += t
        a += entropy((-t).float().exp(), 256)
print(l, a)

In [ ]:
import matplotlib.pyplot as plt
class MyAdd(Add):
    _ = []
    def forward(s, _):
        _ = [s(_) for s in s]
        s._ += [_.norm() for _ in _],
        return reduce(torch.add, _)
class MyRes(Res):
    _ = []
    def forward(s, _):
        t = super().forward(_)
        s._ += [_.norm(), t.norm()],
        return _ + t
for _ in m.modules():
    if isinstance(_, Add):
        _.__class__ = MyAdd
    elif isinstance(_, Res):
        _.__class__ = MyRes
with torch.no_grad():
    m._orig_mod(x)
plt.plot(torch.tensor(MyAdd._).to('cpu', torch.float))
plt.yscale('log')

In [ ]:
plt.plot(torch.tensor(MyRes._).to('cpu',torch.float))
plt.yscale('log')